# LifeMate pipeline: audio → ASR → agent gọi tool → `submission.jsonl`

Chạy trên Kaggle (GPU T4 x2, bật Internet). Kết quả: `/kaggle/working/result/submission.jsonl`.

| Bước | Làm gì | Ý lấy từ paper |
|---|---|---|
| 1. ASR | VietASR (Zipformer 68M, ONNX, CPU) chép audio của lượt cuối và các lượt user trong history; hotword lấy từ user profile + enum trong tool schema | VietASR (Interspeech 2025); Spoken FC (ACM MM 2026): lỗi chính nằm ở tên riêng và số |
| 2. Chuẩn hoá số | "năm trăm nghìn" → 500000; LLM nhận cả bản gốc lẫn bản chuẩn hoá | Spoken FC |
| 3. Few-shot | Lấy k mẫu train có câu lệnh gần nhất (TF-IDF ký tự) làm ví dụ chuỗi tool call đúng, để model học quy ước của gold (vd đọc trước khi ghi) | APIGen-MT: dữ liệu có hành động ground-truth |
| 4. Router | LLM phân loại ACT / ASK / OUT_OF_SCOPE; OUT_OF_SCOPE trả `[]`, không gọi tool nào | When2Call (NAACL 2025), Hammer irrelevance (ICLR 2025) |
| 5. Agent loop | Qwen3-4B-Instruct-2507 gọi tool nhiều bước trên `ToolEnv`; batch theo bước trên vLLM | APIGen-MT / τ-bench: multi-step, chấm theo state |
| 6. Kiểm call trước khi chạy | Ép kiểu nhẹ + `env.validate` (không tính vào bài nộp); call sai không chạy mà trả lỗi để LLM sửa | ToolACE: rule checker; call bị từ chối vẫn tính vào U nếu đã `env.call` |
| 7. Ghi kết quả | `env.get_tool_calls()` + `env.get_state_hash()`; kiểm bằng validator của BTC | README dữ liệu |

**Model** (Apache-2.0, phát hành trước 26/10/2026): `Qwen/Qwen3-4B-Instruct-2507` (~4B) + `zzasdf/viet_iter3_pseudo_label` (68M). Tổng < 6B, LLM < 5B, context 16,384.

**Chạy thử:** để `SPLIT = "train"`, `LIMIT = 30` → xem điểm cục bộ ở mục 12. **Nộp bài:** `SPLIT = "public_test"`, `LIMIT = None`.

**Checkpoint (phiên Kaggle tối đa 12 giờ):** kết quả ghi vào `/kaggle/working/checkpoint/` sau mỗi chunk; sau `TIME_BUDGET_H` giờ notebook dừng nhận chunk mới để kịp lưu output. Lần sau, gắn output của lần trước làm Input: notebook tự tìm thư mục `checkpoint/`, bỏ qua sample đã xong và chạy tiếp. `submission.jsonl` chỉ được ghi khi đủ mọi sample. Đổi `RUN_TAG` khi đổi prompt/model để không dùng lại kết quả cũ.

In [ ]:
# vllm không ghim sẽ kéo torch bản CUDA 13, lệch với torchaudio CUDA 12.8 có sẵn trên Kaggle.
# vllm 0.10.2 ghim torch/torchaudio/torchvision 2.8 cu128 cùng bộ, có wheel cho Python 3.13.
# transformers 5 (Kaggle cài sẵn) ra sau vllm 0.10.2 nên ghim bản cùng thời 4.56.2; bản này cần huggingface_hub < 1.0.
!pip install -q "vllm==0.10.2" "transformers==4.56.2" "huggingface_hub<1.0" sherpa-onnx sentencepiece soundfile librosa scikit-learn
import torch, torchaudio, transformers, vllm
print("vllm", vllm.__version__, "| torch", torch.__version__, "cuda", torch.version.cuda,
      "| torchaudio", torchaudio.__version__, "| transformers", transformers.__version__)
print("GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.__version__.startswith("2.8.0"), "torch cũ vẫn đang nạp: Restart session rồi chạy lại từ ô này"
assert torch.cuda.is_available(), "torch không thấy GPU: kiểm Accelerator = GPU T4 x2"

## 1. Config

In [ ]:
import os, sys, json, re, random, unicodedata, time, subprocess
from pathlib import Path
from collections import Counter

CFG = {
    "SPLIT": "train",                                   # train | public_test | private_test
    "LIMIT": 30,                                        # số sample chạy thử; None = tất cả
    "INPUT_ROOT": "/kaggle/input",
    "DATA_ROOT": None,                                  # None = tự tìm thư mục chứa <split>/<split>.jsonl
    "WORK_DIR": "/kaggle/working",
    "USE_TRAIN_TEXT": False,                            # True: dùng audio_text có sẵn của train thay ASR, để đo riêng phần LLM
    "ASR_BACKEND": "vietasr",                           # vietasr | whisper
    "ASR_HOTWORDS": True,
    "HOTWORDS_SCORE": 1.5,
    "WHISPER_MODEL": "openai/whisper-large-v3-turbo",   # MIT, 809M
    "LLM": "Qwen/Qwen3-4B-Instruct-2507",
    "MAX_MODEL_LEN": 16384,
    "MAX_NEW_TOKENS": 768,
    "MAX_STEPS": 10,
    "MAX_INVALID_RETRIES": 2,
    "TOOL_RESULT_MAX_CHARS": 3000,
    "FEWSHOT_K": 3,
    "USE_ROUTER": False,                                # run v1: router gắn nhầm 2/29 sample trong phạm vi thành OUT_OF_SCOPE
    "ARG_PRUNE": True,                                  # bỏ tham số mà gold train gần như không bao giờ điền (mục 4)
    "SEED": 42,
    "RUN_TAG": "v2",                                    # đổi khi đổi prompt/model; checkpoint khác tag bị bỏ qua
    "CHUNK_SIZE": 64,                                   # số sample mỗi chunk; ghi checkpoint sau mỗi chunk
    "TIME_BUDGET_H": 11.0,                              # dừng nhận chunk mới sau ngần này giờ (phiên Kaggle tối đa 12h)
}
ASK_TOOL = "sys.ask_missing_info"
T_START = time.time()


def find_data_root(split, root, max_depth=5):
    """Vị trí mount của dataset phụ thuộc tên dataset trên Kaggle, nên tìm theo file <split>/<split>.jsonl."""
    base = root.rstrip("/").count("/")
    for cur, dirs, files in os.walk(root):
        if os.path.isfile(os.path.join(cur, split, f"{split}.jsonl")):
            return Path(cur)
        if cur.count("/") - base >= max_depth:
            dirs[:] = []
    raise FileNotFoundError(f"Không thấy {split}/{split}.jsonl dưới {root}; kiểm tra đã Add Input dataset chưa")


ROOT = Path(CFG["DATA_ROOT"]) if CFG["DATA_ROOT"] else find_data_root(CFG["SPLIT"], CFG["INPUT_ROOT"])
SPLIT = CFG["SPLIT"]
SPLIT_DIR = ROOT / SPLIT
WORK = Path(CFG["WORK_DIR"])
OUT_DIR = WORK / "result"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA ROOT:", ROOT)
print((ROOT / "VERSION").read_text() if (ROOT / "VERSION").exists() else "")

random.seed(CFG["SEED"])
os.environ["PYTHONHASHSEED"] = str(CFG["SEED"])
import numpy as np, torch
np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"])

## 2. Dữ liệu, tool catalog và `zaic_tools`
Format theo `README.md` của gói dữ liệu. Agent không đọc `state.json`; mọi dữ liệu app lấy qua `env.call`.

In [ ]:
sys.dont_write_bytecode = True                          # /kaggle/input chỉ đọc
sys.path.insert(0, str(SPLIT_DIR))
from zaic_tools import ToolEnv


def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


SAMPLES = read_jsonl(SPLIT_DIR / f"{SPLIT}.jsonl")
if CFG["LIMIT"]:
    SAMPLES = SAMPLES[: CFG["LIMIT"]]

CATALOG = json.loads((SPLIT_DIR / f"{SPLIT}_tools_catalog.json").read_text(encoding="utf-8"))
TOOL_SPEC = {t["name"]: t for a in CATALOG["apps"] for t in a["tools"]}
APP_INFO = {a["id"]: a for a in CATALOG["apps"]}
FUZZY = {(t["name"], k) for t in TOOL_SPEC.values() for k, v in (t["parameters"].get("properties") or {}).items() if v.get("x-match") == "fuzzy"}

print(f"{len(SAMPLES)} sample | {len(TOOL_SPEC)} tool | {len(FUZZY)} tham số fuzzy")
print(Counter(len(s["history"]) for s in SAMPLES))
if "label" in SAMPLES[0]:
    print(Counter("oos" if not s["label"]["tool_calls"] else ("ask" if any(c["name"] == ASK_TOOL for c in s["label"]["tool_calls"]) else "act") for s in SAMPLES))

## 3. Checkpoint: nạp kết quả của các lần chạy trước
Tìm mọi thư mục `checkpoint/` trong `/kaggle/input` (output notebook lần trước đã gắn làm Input) và trong `/kaggle/working`. Chỉ nhận dòng cùng `SPLIT` và `RUN_TAG`. Kết quả gộp được ghi lại vào `/kaggle/working/checkpoint/`, nên output lần này chứa toàn bộ kết quả cộng dồn.

Checkpoint chỉ lưu sample đã chạy xong. Sample đang chạy dở khi phiên dừng không được lưu, nên lần sau chạy lại từ đầu với một `ToolEnv` mới; không call nào của lần dở dang lọt vào bài nộp.

In [ ]:
CKPT_DIR = WORK / "checkpoint"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RES_FILE = CKPT_DIR / f"{SPLIT}_results.jsonl"
ASR_FILE = CKPT_DIR / f"{SPLIT}_transcripts.json"
TRACE_FILE = CKPT_DIR / f"{SPLIT}_trace.jsonl"


def find_checkpoint_dirs(roots, max_depth=5):
    found = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        base = root.rstrip("/").count("/")
        for cur, dirs, _ in os.walk(root):
            if os.path.basename(cur) == "checkpoint":
                found.append(Path(cur)); dirs[:] = []; continue
            if cur.count("/") - base >= max_depth:
                dirs[:] = []
    return found


DONE, TRANSCRIPTS, TRACES = {}, {}, {}
for d in find_checkpoint_dirs([CFG["INPUT_ROOT"], str(WORK)]):
    n_before = len(DONE)
    if (d / RES_FILE.name).exists():
        for row in read_jsonl(d / RES_FILE.name):
            if row.get("run_tag") == CFG["RUN_TAG"]:
                DONE[row["id"]] = row
    if (d / ASR_FILE.name).exists():
        TRANSCRIPTS.update(json.loads((d / ASR_FILE.name).read_text(encoding="utf-8")))
    if (d / TRACE_FILE.name).exists():
        TRACES.update({r["id"]: r for r in read_jsonl(d / TRACE_FILE.name) if r["id"] in DONE})
    print(f"checkpoint {d}: +{len(DONE) - n_before} sample")

with open(RES_FILE, "w", encoding="utf-8") as f:
    for row in DONE.values():
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
with open(TRACE_FILE, "w", encoding="utf-8") as f:
    for row in TRACES.values():
        f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")
ASR_FILE.write_text(json.dumps(TRANSCRIPTS, ensure_ascii=False), encoding="utf-8")

TODO = [s for s in SAMPLES if s["id"] not in DONE]
print(f"Đã xong {len(SAMPLES) - len(TODO)}/{len(SAMPLES)}, còn {len(TODO)} sample (RUN_TAG={CFG['RUN_TAG']})")

## 4. Tool schema và kiểm call
Ép kiểu nhẹ ("26" → 26) rồi hỏi `env.validate` (validator chính thức, không tính vào bài nộp). Chỉ call qua được mới được `env.call`, vì call bị từ chối vẫn nằm trong `get_tool_calls()` và bị tính vào U.

**Cắt tham số theo thống kê gold train:** call chỉ khớp khi tham số khớp, nên tham số thừa làm mất điểm (vd `alarm.create` kèm `label`, `calendar.list_events` kèm khoảng ngày). Từ nhãn train: tool đọc được gọi với `{}` ở ≥ 80% lần thì luôn gọi `{}`; tham số tuỳ chọn của tool ghi xuất hiện < 5% số lần thì bỏ.

In [ ]:
def _arg_stats(train_rows, min_calls=5):
    n_calls, n_empty, used = Counter(), Counter(), Counter()
    for t in train_rows:
        for c in t["label"]["tool_calls"] or []:
            n_calls[c["name"]] += 1
            n_empty[c["name"]] += not c["arguments"]
            used.update((c["name"], k) for k in c["arguments"])
    read_empty, rare_opt = set(), set()
    for name, n in n_calls.items():
        spec = TOOL_SPEC.get(name)
        if spec is None or n < min_calls:
            continue
        if spec.get("kind") == "read" and n_empty[name] / n >= 0.8:
            read_empty.add(name)
        if spec.get("kind") == "write":
            # tham số thuộc x-one-of là "bắt buộc một trong", bỏ đi có thể làm call thiếu nhóm bắt buộc
            required = set(spec["parameters"].get("required") or []) | {k for g in spec["parameters"].get("x-one-of", []) for k in g}
            rare_opt |= {(name, k) for k in (spec["parameters"].get("properties") or {}) if k not in required and used[(name, k)] / n < 0.05}
    return read_empty, rare_opt


READ_EMPTY, RARE_OPT = _arg_stats(read_jsonl(ROOT / "train" / "train.jsonl")) if CFG["ARG_PRUNE"] else (set(), set())
print(f"{len(READ_EMPTY)} tool đọc luôn gọi {{}} | {len(RARE_OPT)} tham số tuỳ chọn bị bỏ, vd {sorted(RARE_OPT)[:6]}")


def prune_args(name, args):
    if name in READ_EMPTY:
        return {}
    return {k: v for k, v in args.items() if (name, k) not in RARE_OPT}


def tools_for_sample(sample):
    names = [n for app in sample["context"]["installed_apps"].values() for n in app["tools"]] + [ASK_TOOL]
    tools = []
    for n in names:
        spec = TOOL_SPEC.get(n)
        if spec is None:
            continue
        params = {k: v for k, v in spec["parameters"].items() if not k.startswith("x-")}
        desc = spec.get("description", "")
        if "x-one-of" in spec["parameters"]:
            desc += f" (bắt buộc một trong các nhóm: {spec['parameters']['x-one-of']})"
        tools.append({"type": "function", "function": {"name": n, "description": desc, "parameters": params}})
    return tools


def _coerce(value, typ):
    if typ == "integer" and isinstance(value, str) and re.fullmatch(r"-?\d+", value.strip()):
        return int(value.strip())
    if typ == "integer" and isinstance(value, float) and value.is_integer():
        return int(value)
    if typ == "number" and isinstance(value, str) and re.fullmatch(r"-?\d+(\.\d+)?", value.strip()):
        v = float(value.strip())
        return int(v) if v.is_integer() else v
    if typ == "boolean" and isinstance(value, str) and value.lower() in ("true", "false"):
        return value.lower() == "true"
    if typ == "string" and isinstance(value, (int, float)) and not isinstance(value, bool):
        return str(value)
    return value


def check_call(call, env, allowed):
    """Trả về (call đã ép kiểu, lỗi hoặc None)."""
    name = call.get("name")
    if name not in allowed:
        return call, f"Tool '{name}' không có trong các app đã cài của người dùng."
    args = call.get("arguments")
    if isinstance(args, str):
        try:
            args = json.loads(args) if args.strip() else {}
        except json.JSONDecodeError:
            return call, "arguments không phải JSON hợp lệ."
    args = prune_args(name, dict(args or {}))
    props = TOOL_SPEC[name]["parameters"].get("properties") or {}
    for k, v in list(args.items()):
        typ = props.get(k, {}).get("type")
        if isinstance(typ, str):
            args[k] = _coerce(v, typ)
    res = env.validate(name, args)
    if not res.get("ok"):
        err = res.get("error", "lỗi schema")
        if name != ASK_TOOL and "thiếu" in err:
            err += f". Nếu người dùng không cung cấp và không suy ra được từ context/kết quả tool, gọi {ASK_TOOL}."
        return {"name": name, "arguments": args}, err
    return {"name": name, "arguments": args}, None

## 5. ASR
Chép lượt cuối và các lượt user trong history của các sample chưa xong; transcript đã có trong checkpoint thì dùng lại. Chạy trong process riêng để process chính không giữ CUDA trước khi vLLM khởi động. Hotword lấy từ user profile và enum trong schema; không gọi tool trên env để lấy hotword, vì mọi `env.call` đều vào bài nộp.

In [ ]:
def build_hotwords(sample, limit=60):
    words = [v for v in sample["context"].get("user_profile", {}).values() if isinstance(v, str)]
    for t in tools_for_sample(sample):
        for spec in (t["function"]["parameters"].get("properties") or {}).values():
            words += [e.replace("_", " ") for e in spec.get("enum", []) or [] if isinstance(e, str)]
    seen, out = set(), []
    for w in words:
        if 2 <= len(w) <= 40 and w.lower() not in seen:
            seen.add(w.lower()); out.append(w)
    return out[:limit]


manifest = []
for s in TODO:
    hw = build_hotwords(s) if CFG["ASR_HOTWORDS"] else []
    items = [(s["id"], s["audio"])] + [(f"{s['id']}#{k}", t["audio"]) for k, t in enumerate(s["history"]) if t.get("role") == "user" and t.get("audio")]
    manifest += [{"key": key, "audio": str(SPLIT_DIR / a), "hotwords": hw} for key, a in items if key not in TRANSCRIPTS]
(WORK / "asr_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False), encoding="utf-8")
print(len(manifest), "file audio cần chép")

In [ ]:
%%writefile /kaggle/working/asr_run.py
import json, sys
from pathlib import Path
import soundfile as sf, librosa

manifest_path, out_path, backend, hotwords_on, hw_score, whisper_model = sys.argv[1:7]
hotwords_on = hotwords_on == "1"
items = json.loads(Path(manifest_path).read_text(encoding="utf-8"))


def load_audio(path):
    wav, sr = sf.read(path, dtype="float32", always_2d=True)
    wav = wav.mean(axis=1)
    if sr != 16000:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=16000)
    return wav


results = {}
if backend == "vietasr":
    import sherpa_onnx, sentencepiece as spm
    from huggingface_hub import snapshot_download
    d = Path(snapshot_download("zzasdf/viet_iter3_pseudo_label", allow_patterns=[
        "exp/*-epoch-12-avg-8.onnx", "data/Vietnam_bpe_2000_new/*.model", "data/Vietnam_bpe_2000_new/tokens.txt"]))
    lang = d / "data/Vietnam_bpe_2000_new"
    kw = dict(encoder=str(d / "exp/encoder-epoch-12-avg-8.onnx"), decoder=str(d / "exp/decoder-epoch-12-avg-8.onnx"),
              joiner=str(d / "exp/joiner-epoch-12-avg-8.onnx"), tokens=str(lang / "tokens.txt"),
              num_threads=4, sample_rate=16000, feature_dim=80)
    tokens = [l.split()[0] for l in (lang / "tokens.txt").read_text(encoding="utf-8").splitlines() if l.strip()]
    lowercase = not any(ch.isupper() for t in tokens for ch in t)
    use_hw = False
    if hotwords_on:
        # sherpa-onnx cần file vocab "piece<TAB>score"; chỉ bật hotword khi bpe.model khớp tokens.txt
        sp = spm.SentencePieceProcessor(model_file=str(lang / "bpe.model"))
        pieces = [sp.id_to_piece(i) for i in range(sp.get_piece_size())]
        if set(pieces) <= set(tokens):
            vocab = Path("/kaggle/working/bpe.vocab")
            vocab.write_text("\n".join(f"{p}\t{sp.get_score(i)}" for i, p in enumerate(pieces)), encoding="utf-8")
            try:
                rec = sherpa_onnx.OfflineRecognizer.from_transducer(
                    **kw, decoding_method="modified_beam_search", max_active_paths=4,
                    hotwords_file="", hotwords_score=float(hw_score), modeling_unit="bpe", bpe_vocab=str(vocab))
                use_hw = True
            except Exception as e:
                print("[warn] tắt hotword:", e)
        else:
            print("[warn] bpe.model không khớp tokens.txt, tắt hotword")
    if not use_hw:
        rec = sherpa_onnx.OfflineRecognizer.from_transducer(**kw, decoding_method="greedy_search")
    for i, it in enumerate(items):
        hw = "/".join(w.lower() if lowercase else w for w in it["hotwords"]) if use_hw else ""
        try:
            st = rec.create_stream(hotwords=hw) if hw else rec.create_stream()
        except Exception:
            st = rec.create_stream()
        st.accept_waveform(16000, load_audio(it["audio"]))
        rec.decode_stream(st)
        results[it["key"]] = st.result.text.strip()
        if i % 200 == 0:
            print(i, it["key"], results[it["key"]], flush=True)
else:
    import torch
    from transformers import pipeline
    asr = pipeline("automatic-speech-recognition", model=whisper_model, torch_dtype=torch.float16, device=0)
    for it in items:
        out = asr(load_audio(it["audio"]), generate_kwargs={"language": "vi", "task": "transcribe", "num_beams": 1})
        results[it["key"]] = out["text"].strip()

Path(out_path).write_text(json.dumps(results, ensure_ascii=False, indent=1), encoding="utf-8")
print("done", len(results))

In [ ]:
t0 = time.time()
if CFG["USE_TRAIN_TEXT"] and SPLIT == "train":
    for s in TODO:
        TRANSCRIPTS[s["id"]] = s["audio_text"]
        for k, turn in enumerate(s["history"]):
            if turn.get("role") == "user":
                TRANSCRIPTS[f"{s['id']}#{k}"] = turn.get("audio_text", "")
elif manifest:
    subprocess.run([sys.executable, str(WORK / "asr_run.py"), str(WORK / "asr_manifest.json"), str(WORK / "transcripts_new.json"),
                    CFG["ASR_BACKEND"], "1" if CFG["ASR_HOTWORDS"] else "0", str(CFG["HOTWORDS_SCORE"]), CFG["WHISPER_MODEL"]], check=True)
    TRANSCRIPTS.update(json.loads((WORK / "transcripts_new.json").read_text(encoding="utf-8")))
if not (CFG["USE_TRAIN_TEXT"] and SPLIT == "train"):
    ASR_FILE.write_text(json.dumps(TRANSCRIPTS, ensure_ascii=False), encoding="utf-8")
print(f"{len(TRANSCRIPTS)} transcript, {time.time() - t0:.0f}s")
for s in TODO[:5]:
    print(s["id"], "|", TRANSCRIPTS.get(s["id"]), "|", s.get("audio_text", ""))

## 6. Chuẩn hoá số tiếng Việt
Chỉ đổi những cụm chắc chắn là số (có "mươi/trăm/nghìn/triệu/tỷ...", chuỗi ≥ 3 chữ số đọc liền, hoặc một chữ số trước đơn vị như "độ", "giờ").

In [ ]:
_DIGITS = {"không": 0, "một": 1, "mốt": 1, "hai": 2, "ba": 3, "bốn": 4, "tư": 4, "năm": 5, "lăm": 5,
           "sáu": 6, "bảy": 7, "bẩy": 7, "tám": 8, "chín": 9}
_SCALES = {"nghìn": 1000, "ngàn": 1000, "triệu": 10**6, "tỷ": 10**9, "tỉ": 10**9}
_MULT = {"mười", "mươi", "trăm", "linh", "lẻ", "rưỡi"} | set(_SCALES)
_NUMWORDS = set(_DIGITS) | _MULT
_UNITS_AFTER = {"độ", "giờ", "phút", "giây", "người", "phần", "ngày", "tuần", "k", "đồng", "lít", "suất", "phòng", "lần", "chiếc", "cái"}


def _parse_run(toks):
    if all(t in _DIGITS for t in toks):
        return "".join(str(_DIGITS[t]) for t in toks)
    total, small, pending, prev = 0, 0, None, None
    for t in toks:
        if t in _DIGITS:
            if pending is not None:
                return None
            pending = _DIGITS[t]
        elif t == "mười":
            small += 10
        elif t == "mươi":
            small += (pending if pending is not None else 1) * 10; pending = None
        elif t == "trăm":
            small += (pending if pending is not None else 1) * 100; pending = None
        elif t in _SCALES:
            total += (small + (pending or 0) or 1) * _SCALES[t]; small, pending = 0, None
        elif t == "rưỡi":
            if prev == "trăm":
                small += 50
            elif prev in _SCALES:
                total += _SCALES[prev] // 2
            else:
                return None
        prev = t
    # nói tắt: "một triệu hai" = 1,200,000; "hai trăm năm" = 250
    if pending is not None and len(toks) >= 2 and toks[-2] in _SCALES and small == 0:
        return str(total + pending * _SCALES[toks[-2]] // 10)
    if pending is not None and len(toks) >= 2 and toks[-2] == "trăm":
        return str(total + small + pending * 10)
    return str(total + small + (pending or 0))


def normalize_numbers(text):
    toks = text.split()
    low = [unicodedata.normalize("NFC", t.lower().strip(",.!?")) for t in toks]
    out, i = [], 0
    while i < len(toks):
        if low[i] in _NUMWORDS and low[i] not in ("linh", "lẻ", "rưỡi"):
            j = i
            while j < len(toks) and low[j] in _NUMWORDS:
                j += 1
            run, nxt = low[i:j], (low[j] if j < len(toks) else "")
            convert = any(t in _MULT for t in run) or len(run) >= 3 or (len(run) == 1 and nxt in _UNITS_AFTER and run[0] != "không")
            val = _parse_run(run) if convert else None
            if val is not None:
                out.append(val); i = j; continue
        out.append(toks[i]); i += 1
    return " ".join(out)


_H = r"(\d{1,2}|mười một|mười hai|một|hai|ba|bốn|năm|sáu|bảy|tám|chín|mười)"
_PERIOD = r"(?:\s+(sáng|trưa|chiều|tối|đêm))?"
_TIME_RES = [
    (re.compile(rf"\b{_H}\s+giờ\s+kém\s+(\d{{1,2}})(?:\s+phút)?{_PERIOD}"), lambda h, m: (h - 1, 60 - m)),
    (re.compile(rf"\b{_H}(?:\s+giờ)?\s+rưỡi{_PERIOD}"), lambda h, m: (h, 30)),
    (re.compile(rf"\b{_H}\s+giờ\s+(\d{{1,2}})(?:\s+phút)?{_PERIOD}"), lambda h, m: (h, m)),
    (re.compile(rf"\b{_H}\s+giờ{_PERIOD}(?!\s*(?:đồng hồ|nữa|sau|trước))"), lambda h, m: (h, 0)),
]


def _hour_value(tok):
    return int(tok) if tok.isdigit() else int(_parse_run(tok.split()))


def _to_24h(h, period):
    if period in ("chiều", "tối") and h < 12:
        return h + 12
    if period == "trưa" and h < 11:
        return h + 12
    if period == "đêm" and 6 <= h < 12:
        return h + 12
    return h


def normalize_time(text):
    """'8 giờ kém 15 tối' → '19:45 tối', 'tám rưỡi sáng' → '08:30 sáng'. Chạy sau normalize_numbers."""
    text = unicodedata.normalize("NFC", text.lower())
    for rx, fn in _TIME_RES:
        def sub(m):
            if re.search(r"\b(trong|sau|mất|khoảng|thêm)\s*$", m.string[: m.start()]):
                return m.group(0)                       # "trong 2 giờ" là khoảng thời gian, không phải giờ đồng hồ
            g = m.groups()
            h = _hour_value(g[0])
            minute = int(g[1]) if len(g) == 3 and g[1] and g[1].isdigit() else 0
            h, minute = fn(h, minute)
            period = g[-1]
            if not (0 <= h <= 24 and 0 <= minute < 60):
                return m.group(0)
            return f"{_to_24h(h, period):02d}:{minute:02d}" + (f" {period}" if period else "")
        text = rx.sub(sub, text)
    return text


for s in ["chuyển cho mẹ năm trăm nghìn", "bật điều hòa hai mươi sáu độ", "nhắc tôi lúc ba giờ chiều mai", "chuyển một triệu rưỡi", "tắt đèn tầng một",
          "NHẮC CHỊ RA SÂN BAY LÚC TÁM GIỜ KÉM MƯỜI LĂM TỐI MAI", "GỌI LẠI CHO KHÁCH LÚC TÁM RƯỠI TỐI MAI", "BÁO THỨC NĂM RƯỠI SÁNG MAI", "họp lúc hai giờ ba mươi chiều"]:
    print(f"{s!r:58} → {normalize_time(normalize_numbers(s))!r}")

## 7. Few-shot từ train
Với mỗi sample, lấy `FEWSHOT_K` mẫu train có câu lệnh gần nhất (TF-IDF n-gram ký tự) và đưa chuỗi tool call đúng của chúng vào prompt. Khi chạy trên train thì bỏ chính sample đó.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

TRAIN = read_jsonl(ROOT / "train" / "train.jsonl")
_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True)
_train_mat = _vec.fit_transform([t["audio_text"] for t in TRAIN])


def _fmt_calls(calls):
    # Viết đúng format <tool_call> của Qwen: run v1 hiển thị dạng name({...}) và model chép y nguyên dạng đó thay vì gọi tool.
    if not calls:
        return "OUT_OF_SCOPE"
    return "\n".join(f'<tool_call>\n{json.dumps({"name": c["name"], "arguments": c["arguments"]}, ensure_ascii=False)}\n</tool_call>' for c in calls)


def fewshot_block(sample, query):
    if CFG["FEWSHOT_K"] <= 0:
        return ""
    sims = (_train_mat @ _vec.transform([query]).T).toarray().ravel()
    picked = []
    for i in sims.argsort()[::-1]:
        if TRAIN[i]["id"] != sample["id"]:
            picked.append(TRAIN[i])
        if len(picked) == CFG["FEWSHOT_K"]:
            break
    parts = [f'Ví dụ {n + 1}: "{t["audio_text"]}"\n{_fmt_calls(t["label"]["tool_calls"])}' for n, t in enumerate(picked)]
    return ("Các ví dụ dưới đây là của người dùng KHÁC (ID và dữ liệu khác), chỉ để tham khảo cách chọn tool, thứ tự tool và "
            "những tham số nào được điền. Ví dụ liệt kê gộp mọi call; khi chạy thật, mỗi lượt bạn chỉ gọi các tool cần ngay lúc đó:\n\n"
            + "\n\n".join(parts))

## 8. Prompt

In [ ]:
SYSTEM_RULES = f"""Bạn là trợ lý giọng nói điều khiển các app của người dùng bằng cách gọi tool. Bạn chỉ hành động qua tool call; không cần trả lời bằng lời.

Quy tắc:
1. Chỉ dùng các tool được cung cấp, mỗi call viết trong <tool_call>{{"name": ..., "arguments": {{...}}}}</tool_call>. Nếu ngay từ đầu không app nào xử lý được yêu cầu, KHÔNG gọi tool nào (kể cả tool đọc), chỉ trả lời: OUT_OF_SCOPE. Đã gọi tool rồi thì không trả lời OUT_OF_SCOPE nữa.
2. Nếu thiếu tham số bắt buộc mà không suy ra được từ yêu cầu, context, user profile, lịch sử hoặc kết quả tool, gọi {ASK_TOOL} với tool_name là tool định gọi và params là tên các tham số còn thiếu (đúng tên trong schema), rồi dừng. Không bịa giá trị. Thông tin tra được bằng tool đọc (số điện thoại, ID, số tiền hoá đơn, địa điểm trong lịch) thì tra, không hỏi.
3. Khi cần ID hay dữ liệu hiện tại (contact_id, device_id, event_id, số dư...), gọi tool đọc trước rồi dùng kết quả cho tool ghi. Làm theo cách sắp xếp tool trong các ví dụ. Không gọi tool đọc không cần cho yêu cầu, không gọi lại tool đọc đã có kết quả.
4. Chỉ điền tham số bắt buộc và tham số người dùng nói rõ. Không tự thêm nhãn, ghi chú, bộ lọc, khoảng ngày. Tool đọc dạng liệt kê thì gọi không tham số.
5. Yêu cầu có điều kiện ("nếu ... thì ..."): gọi tool đọc để kiểm điều kiện, đọc kỹ kết quả; chỉ gọi tool ghi khi điều kiện ĐÚNG. Điều kiện sai thì dừng (DONE). Người dùng đổi ý giữa câu ("à không", "mà thôi") thì làm theo ý cuối.
6. Chép nguyên giá trị từ kết quả tool và context (ID, địa chỉ, tên). Số ghi bằng số đúng kiểu schema. Thời gian quy đổi theo current_time và timezone, định dạng giống ví dụ (vd 2026-11-29T09:00:00+07:00); dùng bản đã chuẩn hoá giờ 24h nếu có.
7. Bản chép giọng nói (ASR) không dấu câu và có thể sai chính tả, tên riêng hoặc số. Khi tên gần giống một thực thể trong context hoặc kết quả tool, dùng tên đúng của thực thể đó.
8. Khi đã gọi đủ tool để hoàn thành yêu cầu, trả lời ngắn: DONE."""


def build_system(sample, query):
    ctx = {k: v for k, v in sample["context"].items() if k != "installed_apps"}
    apps = "\n".join(f"- {a}: {APP_INFO.get(a, {}).get('name_vi', '')}. {APP_INFO.get(a, {}).get('description', '')}" for a in sample["context"]["installed_apps"])
    return (f"{SYSTEM_RULES}\n\nContext:\n{json.dumps(ctx, ensure_ascii=False, indent=1)}\n\nApp đã cài:\n{apps}\n\n{fewshot_block(sample, query)}")


def build_user_turn(transcript):
    norm = normalize_time(normalize_numbers(transcript))
    msg = f'Yêu cầu (bản chép ASR, có thể có lỗi): "{transcript}"'
    if norm != unicodedata.normalize("NFC", transcript.lower()):
        msg += f'\nBản đã chuẩn hoá số và giờ (24h): "{norm}"'
    return msg


def history_messages(sample):
    msgs = []
    for k, turn in enumerate(sample["history"]):
        if turn.get("role") == "user":
            msgs.append({"role": "user", "content": build_user_turn(TRANSCRIPTS.get(f"{sample['id']}#{k}", ""))})
        else:
            calls = turn.get("tool_calls") or []
            if calls:
                msgs.append({"role": "assistant", "content": "", "tool_calls": [{"type": "function", "function": {"name": c["name"], "arguments": c["arguments"]}} for c in calls]})
                for r in (turn.get("tool_results") or [])[: len(calls)]:
                    msgs.append({"role": "tool", "content": json.dumps(r, ensure_ascii=False)[: CFG["TOOL_RESULT_MAX_CHARS"]]})
            msgs.append({"role": "assistant", "content": turn.get("text", "")})
    return msgs


ROUTER_PROMPT = """Phân loại yêu cầu CUỐI của người dùng với các app đã cài. Trả lời đúng một nhãn:
- ACT: các app đã cài làm được (kể cả khi phải đọc dữ liệu trước).
- ASK: có app phù hợp nhưng người dùng chưa nói đủ thông tin bắt buộc.
- OUT_OF_SCOPE: không app đã cài nào làm được yêu cầu này.

App đã cài:
{apps}
{history}
{user}

Nhãn:"""

## 9. LLM (vLLM, tensor parallel trên 2 GPU)
Bỏ qua khi checkpoint đã đủ mọi sample.

In [ ]:
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
# T4 (SM 7.5): FlashAttention và nhánh paged của xformers đều cần SM 8.0+.
# Kernel Triton unified không kiểm SM và là backend vllm bản mới tự chọn cho T4.
if torch.cuda.get_device_capability(0) < (8, 0):
    os.environ.setdefault("VLLM_ATTENTION_BACKEND", "TRITON_ATTN_VLLM_V1")
print("attention backend:", os.environ.get("VLLM_ATTENTION_BACKEND", "mặc định"))
from vllm import LLM, SamplingParams

if TODO:
    llm = LLM(model=CFG["LLM"], dtype="half", tensor_parallel_size=max(1, torch.cuda.device_count()),
              max_model_len=CFG["MAX_MODEL_LEN"], seed=CFG["SEED"],
              # Mặc định 256 seq: lúc warm-up sampler sort logits 256 x 151k vocab, vượt phần VRAM còn lại của T4.
              # Mỗi lần generate chỉ gửi tối đa một chunk, nên max_num_seqs = CHUNK_SIZE là đủ.
              max_num_seqs=CFG["CHUNK_SIZE"], gpu_memory_utilization=0.85)
    TOK = llm.get_tokenizer()
    # Lỗi kernel attention chỉ lộ ra ở lần generate đầu, nên chạy thử ngay tại đây với batch lẫn prompt dài/ngắn
    # (giống lúc chạy thật) thay vì để engine chết giữa chunk.
    _t = time.time()
    _out = llm.generate(["xin chào " * 1500, "Trả lời ngắn: 1+1=?", "ok " * 300],
                        SamplingParams(temperature=0.0, max_tokens=16), use_tqdm=False)
    print(f"smoke test ok ({time.time() - _t:.1f}s):", repr(_out[1].outputs[0].text))
SP_AGENT = SamplingParams(temperature=0.0, max_tokens=CFG["MAX_NEW_TOKENS"], seed=CFG["SEED"])
SP_ROUTER = SamplingParams(temperature=0.0, max_tokens=8, seed=CFG["SEED"])


def render(messages, tools=None):
    kw = dict(tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return TOK.apply_chat_template(messages, tools=tools, **kw) if tools else TOK.apply_chat_template(messages, **kw)


_TC_RE = re.compile(r"<tool_call>(.*?)(?:</tool_call>|$)", re.S)
# Run v1: model đôi khi viết call trần dạng `weather.get_forecast({"date": ...})` hoặc `sys.ask_missing_info{...}`
_BARE_RE = re.compile(r"(?<![\w.])([a-z][a-z0-9_]*\.[a-z][a-z0-9_]*)\s*\(?\s*(?=\{)")
_JSON = json.JSONDecoder()


def _decode_at(text, pos):
    try:
        return _JSON.raw_decode(text, pos)[0]
    except json.JSONDecodeError:
        return None


def parse_tool_calls(text):
    calls = []
    for m in _TC_RE.finditer(text):
        body = m.group(1)
        start = body.find("{")
        obj = _decode_at(body, start) if start >= 0 else None
        if not isinstance(obj, dict):
            calls.append({"name": None, "arguments": {}}); continue
        calls.append({"name": obj.get("name"), "arguments": obj.get("arguments", obj.get("parameters", {}))})
    if calls:
        return calls
    for m in _BARE_RE.finditer(text):
        if m.group(1) not in TOOL_SPEC:
            continue
        args = _decode_at(text, m.end())
        ok = isinstance(args, dict)
        calls.append({"name": m.group(1) if ok else None, "arguments": args if ok else {}})
    return calls

## 10. Agent loop theo chunk, ghi checkpoint sau mỗi chunk
Mỗi `ToolEnv` chỉ tạo một lần cho mỗi sample. Trong một chunk: router → vòng lặp batch theo bước (render prompt → sinh một lượt → kiểm call → `env.call` call hợp lệ → thêm kết quả vào hội thoại). Hết chunk thì ghi kết quả vào checkpoint. Quá `TIME_BUDGET_H` thì không nhận chunk mới.

In [ ]:
def _short(obj, n):
    s = obj if isinstance(obj, str) else json.dumps(obj, ensure_ascii=False, default=str)
    return s if len(s) <= n else s[:n] + "...(cắt bớt)"


def _fit_context(st):
    """Cắt bớt kết quả tool khi prompt vượt context 16K."""
    budget = CFG["MAX_MODEL_LEN"] - CFG["MAX_NEW_TOKENS"] - 16
    limit = CFG["TOOL_RESULT_MAX_CHARS"]
    prompt = render(st["messages"], st["tools"])
    while len(TOK(prompt).input_ids) > budget and limit > 200:
        limit //= 2
        for m in st["messages"]:
            if m["role"] == "tool":
                m["content"] = _short(m["content"], limit)
        prompt = render(st["messages"], st["tools"])
    return prompt if len(TOK(prompt).input_ids) <= budget else None


def init_state(s):
    query = TRANSCRIPTS.get(s["id"], "")
    tools = tools_for_sample(s)
    messages = [{"role": "system", "content": build_system(s, query)}] + history_messages(s) + [{"role": "user", "content": build_user_turn(query)}]
    return {"id": s["id"], "sample": s, "env": ToolEnv(SPLIT, s["id"], root=str(ROOT)), "tools": tools,
            "allowed": {t["function"]["name"] for t in tools}, "messages": messages, "query": query,
            "done": False, "steps": 0, "retries": 0, "route": None, "trace": []}


def route(states):
    prompts = []
    for st in states:
        apps = "\n".join(f"- {a}: {', '.join(v['tools'])}" for a, v in st["sample"]["context"]["installed_apps"].items())
        hist = "\n".join(m["content"] for m in history_messages(st["sample"]) if m["role"] == "user" or (m["role"] == "assistant" and m["content"]))
        hist = f"\nLịch sử hội thoại:\n{hist}\n" if hist else ""
        prompts.append(render([{"role": "user", "content": ROUTER_PROMPT.format(apps=apps, history=hist, user=build_user_turn(st["query"]))}]))
    for st, o in zip(states, llm.generate(prompts, SP_ROUTER, use_tqdm=False)):
        label = o.outputs[0].text.strip().upper()
        st["route"] = "OUT_OF_SCOPE" if "OUT" in label else ("ASK" if label.startswith("ASK") else "ACT")
        st["done"] = st["route"] == "OUT_OF_SCOPE"


def agent_step(st, text):
    st["trace"].append(text)
    st["steps"] += 1
    calls = parse_tool_calls(text)
    if not calls:
        st["done"] = True; return

    checked = [check_call(c, st["env"], st["allowed"]) if c.get("name") else (c, "Không đọc được JSON của tool call.") for c in calls]
    errors = [e for _, e in checked if e]
    if errors and st["retries"] < CFG["MAX_INVALID_RETRIES"]:
        st["retries"] += 1
        st["messages"].append({"role": "assistant", "content": text})
        st["messages"].append({"role": "user", "content": "Các tool call trên CHƯA được thực thi vì lỗi:\n- " + "\n- ".join(errors) + "\nHãy gửi lại toàn bộ tool call đã sửa."})
        return

    done_keys = {json.dumps([c["name"], c["arguments"]], sort_keys=True, ensure_ascii=False) for c in st["env"].get_tool_calls()}
    fresh = [c for c, e in checked if not e and json.dumps([c["name"], c["arguments"]], sort_keys=True, ensure_ascii=False) not in done_keys]
    if not fresh:
        st["done"] = True; return

    st["messages"].append({"role": "assistant", "content": "", "tool_calls": [{"type": "function", "function": {"name": c["name"], "arguments": c["arguments"]}} for c in fresh]})
    for c in fresh:
        st["messages"].append({"role": "tool", "content": _short(st["env"].call(c["name"], c["arguments"]), CFG["TOOL_RESULT_MAX_CHARS"])})
    if any(c["name"] == ASK_TOOL for c in fresh) or st["steps"] >= CFG["MAX_STEPS"]:
        st["done"] = True


def run_chunk(samples):
    states = [init_state(s) for s in samples]
    if CFG["USE_ROUTER"]:
        route(states)
    while True:
        prompts, runnable = [], []
        for st in states:
            if st["done"]:
                continue
            p = _fit_context(st)
            if p is None:
                st["done"] = True; continue
            prompts.append(p); runnable.append(st)
        if not runnable:
            break
        for st, o in zip(runnable, llm.generate(prompts, SP_AGENT, use_tqdm=False)):
            agent_step(st, o.outputs[0].text)
    return states


def save_chunk(states):
    with open(RES_FILE, "a", encoding="utf-8") as fr, open(TRACE_FILE, "a", encoding="utf-8") as ft:
        for st in states:
            row = {"id": st["id"], "run_tag": CFG["RUN_TAG"], "tool_calls": st["env"].get_tool_calls(), "state_hash": st["env"].get_state_hash(),
                   "route": st["route"], "steps": st["steps"], "retries": st["retries"], "query": st["query"]}
            DONE[st["id"]] = row
            fr.write(json.dumps(row, ensure_ascii=False) + "\n")
            ft.write(json.dumps({"id": st["id"], "llm_outputs": st["trace"]}, ensure_ascii=False, default=str) + "\n")


budget_s = CFG["TIME_BUDGET_H"] * 3600
chunks = [TODO[i: i + CFG["CHUNK_SIZE"]] for i in range(0, len(TODO), CFG["CHUNK_SIZE"])]
for ci, chunk in enumerate(chunks):
    elapsed = time.time() - T_START
    if elapsed > budget_s:
        print(f"Hết ngân sách thời gian ({elapsed / 3600:.2f}h); dừng ở chunk {ci}/{len(chunks)}")
        break
    t0 = time.time()
    save_chunk(run_chunk(chunk))
    print(f"chunk {ci + 1}/{len(chunks)}: {len(chunk)} sample, {time.time() - t0:.0f}s | xong {len(DONE)}/{len(SAMPLES)} | tổng {(time.time() - T_START) / 3600:.2f}h", flush=True)

## 11. Ghi submission (khi đủ mọi sample) và kiểm bằng validator của BTC

In [ ]:
missing = [s["id"] for s in SAMPLES if s["id"] not in DONE]
sub_path = OUT_DIR / "submission.jsonl"
if missing:
    print(f"CHƯA ĐỦ: còn {len(missing)}/{len(SAMPLES)} sample. Lưu Version, rồi chạy lần sau với output này gắn làm Input để chạy tiếp.")
    if sub_path.exists():
        sub_path.unlink()
else:
    with open(sub_path, "w", encoding="utf-8") as f:
        for s in SAMPLES:
            row = DONE[s["id"]]
            f.write(json.dumps({"id": s["id"], "tool_calls": row["tool_calls"], "state_hash": row["state_hash"]}, ensure_ascii=False) + "\n")
    print(sub_path, "|", len(SAMPLES), "dòng")
    print(Counter("oos" if not DONE[s["id"]]["tool_calls"] else ("ask" if any(c["name"] == ASK_TOOL for c in DONE[s["id"]]["tool_calls"]) else "act") for s in SAMPLES))
    r = subprocess.run([sys.executable, "-m", "zaic_tools", "--split", SPLIT, "--root", "..", "--validate", str(sub_path)],
                       cwd=SPLIT_DIR, capture_output=True, text=True, env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"})
    print(r.stdout[-2000:], r.stderr[-2000:])

## 12. Chấm cục bộ (split `train`)
Theo công thức trong rule: gold rỗng → 1 khi nộp rỗng; sai hash → 0; còn lại F1 với P = M/(M+U), R = M/|G|. Tham số `x-match: fuzzy` so bằng token-F1 > 0.6 sau lowercase + NFC. Hash gold tính bằng cách chạy lại call gold trên một `ToolEnv` mới; việc này chỉ để chấm cục bộ.

In [ ]:
def _norm(s):
    return unicodedata.normalize("NFC", str(s)).lower()


def token_f1(a, b):
    ta, tb = _norm(a).split(), _norm(b).split()
    if not ta or not tb:
        return float(ta == tb)
    common = sum((Counter(ta) & Counter(tb)).values())
    if common == 0:
        return 0.0
    p, r = common / len(ta), common / len(tb)
    return 2 * p * r / (p + r)


def call_match(pred, gold):
    if pred.get("name") != gold.get("name"):
        return False
    pa, ga = pred.get("arguments") or {}, gold.get("arguments") or {}
    if set(pa) != set(ga):
        return False
    return all(token_f1(pa[k], ga[k]) > 0.6 if (gold["name"], k) in FUZZY else pa[k] == ga[k] for k in ga)


def sample_score(pred_calls, gold_calls, pred_hash, gold_hash):
    if not gold_calls:
        return 1.0 if not pred_calls else 0.0
    if pred_hash != gold_hash:
        return 0.0
    used, m = set(), 0
    for g in gold_calls:
        for i, p in enumerate(pred_calls):
            if i not in used and call_match(p, g):
                used.add(i); m += 1; break
    if m == 0:
        return 0.0
    prec, rec = m / len(pred_calls), m / len(gold_calls)
    return 2 * prec * rec / (prec + rec)


if SPLIT == "train":
    rows = []
    for s in SAMPLES:
        if s["id"] not in DONE:
            continue
        done = DONE[s["id"]]
        gold_calls = s["label"]["tool_calls"]
        genv = ToolEnv(SPLIT, s["id"], root=str(ROOT))
        for c in gold_calls:
            genv.call(c["name"], c["arguments"])
        sc = sample_score(done["tool_calls"], gold_calls, done["state_hash"], genv.get_state_hash())
        kind = "oos" if not gold_calls else ("ask" if any(c["name"] == ASK_TOOL for c in gold_calls) else "act")
        rows.append({"id": s["id"], "kind": kind, "score": sc, "hash_ok": done["state_hash"] == genv.get_state_hash(),
                     "route": done["route"], "pred": done["tool_calls"], "gold": gold_calls, "query": done["query"], "audio_text": s["audio_text"]})
    (OUT_DIR / "train_eval.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in rows), encoding="utf-8")
if SPLIT == "train" and rows:
    print(f"Điểm cục bộ: {100 * sum(r['score'] for r in rows) / len(rows):.2f} trên {len(rows)} sample")
    for kind in ("act", "ask", "oos"):
        ks = [r for r in rows if r["kind"] == kind]
        if ks:
            print(f"  {kind}: {100 * sum(r['score'] for r in ks) / len(ks):.2f} (n={len(ks)}, hash đúng {sum(r['hash_ok'] for r in ks)})")
    print("\nSample sai:")
    for r in [r for r in rows if r["score"] < 1][:8]:
        print(f"\n{r['id']} [{r['kind']}] score={r['score']:.2f} route={r['route']}\n  ASR : {r['query']}\n  text: {r['audio_text']}\n  pred: {_fmt_calls(r['pred'])}\n  gold: {_fmt_calls(r['gold'])}")